# U-MIMIC Validation Figures

This notebook generates all validation figures for the U-MIMIC manuscript.

**Validation approach:**
1. **Real data**: Fit U-MIMIC to PhenoPop Ba/F3 imatinib data (sensitive + resistant)
2. **Model comparison**: Cytotoxic vs cytostatic mechanism
3. **Synthetic recovery**: Parameter recovery benchmark with known ground truth
4. **Biological plausibility**: Compare estimated parameters to literature values

**Dataset**: Wu et al., PLoS Comp Bio 2024 (PhenoPop)
- Cell line: Ba/F3 (murine pro-B, BCR-ABL transformed)
- Drug: Imatinib (tyrosine kinase inhibitor)
- 11 concentrations (0–5 µM), 14 time points (0–39h, every 3h)

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

%matplotlib inline

# Publication-quality settings
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'legend.fontsize': 8,
    'figure.figsize': (12, 8),
})

from umimic.data.public_datasets import load_phenopop, list_available_datasets
from umimic.data.schemas import TimeSeriesData, ExperimentalDataset
from umimic.dynamics.states import CellType, ModelTopology
from umimic.dynamics.rates import RateSet, EmaxHill
from umimic.dynamics.ode_system import CellDynamicsODE
from umimic.dynamics.moment_equations import MomentODE
from umimic.inference.likelihood import ModelLikelihood
from umimic.inference.mle import MLEstimator
from umimic.observations.cell_counts import CellCountObservation

print('All imports successful!')

---
## Helper Functions

In [ ]:
def group_by_concentration(dataset):
    """Group series by concentration, harmonize timepoint lengths."""
    conc_groups = {}
    for s in dataset.series:
        if s.concentration is None:
            continue
        conc_key = round(s.concentration, 4)
        if conc_key not in conc_groups:
            conc_groups[conc_key] = []
        conc_groups[conc_key].append(s)

    harmonized = {}
    for conc, series_list in sorted(conc_groups.items()):
        lengths = [len(s.times) for s in series_list]
        target_len = max(set(lengths), key=lengths.count)
        harmonized[conc] = [s for s in series_list if len(s.times) == target_len]
    return harmonized


def fit_per_concentration(conc_groups, n_reps=5, n_restarts=3):
    """Fit simple birth-death model at each concentration independently."""
    topo = ModelTopology(
        active_states=[CellType.P], transitions=[],
        division_states=[CellType.P], death_states=[CellType.P],
    )
    param_names = ['b0', 'd0_P', 'overdispersion']
    results = {}

    for conc, series_list in conc_groups.items():
        data_list = series_list[:n_reps]
        obs_model = CellCountObservation(overdispersion=10.0, count_type='total')
        likelihood = ModelLikelihood(
            topology=topo, data=data_list, param_names=param_names,
            mode='ode', observation_model=obs_model,
        )
        estimator = MLEstimator(
            likelihood,
            bounds={'b0': (1e-4, 0.15), 'd0_P': (1e-5, 0.15), 'overdispersion': (1.0, 500.0)},
            method='L-BFGS-B',
        )
        results[conc] = estimator.fit(n_restarts=n_restarts)
        p = results[conc].parameters
        net = p.get('b0', 0) - p.get('d0_P', 0)
        print(f'  C={conc:6.3f}: net={net:+.4f}/h, LL={results[conc].log_likelihood:.1f}')

    return results


def fit_dose_response(conc_groups, n_reps=3, n_restarts=5):
    """Fit full Emax-Hill dose-response model across all concentrations."""
    topo = ModelTopology.two_state()
    param_names = ['b0', 'd0_P', 'emax_death', 'ec50_death', 'hill_death',
                   'u_PQ', 'u_QP', 'overdispersion']
    all_series = []
    for sl in conc_groups.values():
        all_series.extend(sl[:n_reps])

    obs_model = CellCountObservation(overdispersion=10.0)
    likelihood = ModelLikelihood(
        topology=topo, data=all_series, param_names=param_names,
        mode='ode', observation_model=obs_model,
    )
    estimator = MLEstimator(
        likelihood,
        bounds={'b0': (0.01, 0.12), 'd0_P': (1e-4, 0.05),
                'emax_death': (0.001, 0.3), 'ec50_death': (0.01, 5.0),
                'hill_death': (0.3, 5.0), 'u_PQ': (1e-5, 0.02),
                'u_QP': (1e-5, 0.02), 'overdispersion': (2.0, 500.0)},
        method='L-BFGS-B',
    )
    initial = np.array([0.04, 0.005, 0.05, 0.5, 1.5, 0.003, 0.002, 50.0])
    result = estimator.fit(initial_guess=initial, n_restarts=n_restarts)
    print(f'  Converged: {result.converged}, LL: {result.log_likelihood:.1f}, AIC: {result.aic:.1f}')
    return result


def build_rate_set(params):
    """Build RateSet from parameter dict."""
    death_mod = None
    if params.get('emax_death', 0) > 0.001:
        death_mod = EmaxHill(
            emax=params['emax_death'], ec50=params['ec50_death'],
            hill=params['hill_death'],
        )
    return RateSet(
        birth_base=params['b0'],
        death_base={CellType.P: params['d0_P'], CellType.Q: params['d0_P'] * 0.5},
        death_modulation={CellType.P: death_mod} if death_mod else {},
        transition_base={
            (CellType.P, CellType.Q): params.get('u_PQ', 0.003),
            (CellType.Q, CellType.P): params.get('u_QP', 0.002),
        },
    )

print('Helper functions defined.')

---
## 1. Load PhenoPop Data

In [ ]:
# Load sensitive and resistant populations
sens_dataset = load_phenopop(population='sensitive')
res_dataset = load_phenopop(population='resistant')

sens_groups = group_by_concentration(sens_dataset)
res_groups = group_by_concentration(res_dataset)

print(f'Sensitive: {sens_dataset.n_series} series, {len(sens_groups)} concentrations')
print(f'Resistant: {res_dataset.n_series} series, {len(res_groups)} concentrations')
print(f'\nConcentrations (uM): {sorted(sens_groups.keys())}')

# Data summary table
print(f"\n{'Conc (uM)':>10s} | {'Sens reps':>10s} | {'Sens N0':>10s} | {'Sens Nf':>10s} | {'Res reps':>10s} | {'Res Nf':>10s}")
print('-' * 75)
for conc in sorted(sens_groups.keys()):
    s_sl = sens_groups.get(conc, [])
    r_sl = res_groups.get(conc, [])
    s_n0 = np.mean([s.observations['cell_counts'][0] for s in s_sl]) if s_sl else 0
    s_nf = np.mean([s.observations['cell_counts'][-1] for s in s_sl]) if s_sl else 0
    r_nf = np.mean([s.observations['cell_counts'][-1] for s in r_sl]) if r_sl else 0
    print(f'{conc:10.3f} | {len(s_sl):10d} | {s_n0:10.0f} | {s_nf:10.0f} | {len(r_sl):10d} | {r_nf:10.0f}')

---
## Figure 1: Raw Data Overview

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
s_concs = sorted(sens_groups.keys())
r_concs = sorted(res_groups.keys())
colors = plt.cm.viridis(np.linspace(0, 0.95, len(s_concs)))

# Panel A: Sensitive mean trajectories
ax = axes[0, 0]
for conc, color in zip(s_concs, colors):
    all_c = [s.observations['cell_counts'] for s in sens_groups[conc]]
    mean_c = np.mean(all_c, axis=0)
    t = sens_groups[conc][0].times
    ax.plot(t, mean_c, color=color, linewidth=1.5,
            label=f'{conc}' if conc in [0, 0.5, 5.0] else None)
ax.set_xlabel('Time (h)')
ax.set_ylabel('Cell count')
ax.set_title('A. Sensitive Ba/F3 (mean)')
ax.legend(title='uM', fontsize=7)

# Panel B: Resistant mean trajectories
ax = axes[0, 1]
for conc, color in zip(r_concs, colors[:len(r_concs)]):
    all_c = [s.observations['cell_counts'] for s in res_groups[conc]]
    mean_c = np.mean(all_c, axis=0)
    t = res_groups[conc][0].times
    ax.plot(t, mean_c, color=color, linewidth=1.5,
            label=f'{conc}' if conc in [0, 0.5, 5.0] else None)
ax.set_xlabel('Time (h)')
ax.set_ylabel('Cell count')
ax.set_title('B. Resistant Ba/F3 (mean)')
ax.legend(title='uM', fontsize=7)

# Panel C: Sensitive with replicates (select concentrations)
ax = axes[0, 2]
for conc_target, pc in zip([0, 0.25, 1.25, 5.0],
                            ['#2196F3', '#4CAF50', '#FF9800', '#F44336']):
    ckey = min(s_concs, key=lambda c: abs(c - conc_target))
    for s in sens_groups[ckey][:5]:
        ax.plot(s.times, s.observations['cell_counts'], color=pc, alpha=0.2, linewidth=0.7)
    mean_c = np.mean([s.observations['cell_counts'] for s in sens_groups[ckey]], axis=0)
    ax.plot(sens_groups[ckey][0].times, mean_c, color=pc, linewidth=2,
            label=f'{ckey} uM')
ax.set_xlabel('Time (h)')
ax.set_ylabel('Cell count')
ax.set_title('C. Sensitive: Select Doses + Replicates')
ax.legend(fontsize=8)

# Panel D: Fold change from t=0 (sensitive)
ax = axes[1, 0]
for conc, color in zip(s_concs, colors):
    all_fc = []
    for s in sens_groups[conc]:
        counts = s.observations['cell_counts']
        if counts[0] > 0:
            all_fc.append(counts / counts[0])
    if all_fc:
        mean_fc = np.mean(all_fc, axis=0)
        ax.plot(sens_groups[conc][0].times, mean_fc, color=color, linewidth=1.2)
ax.axhline(y=1, color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('Time (h)')
ax.set_ylabel('Fold change from t=0')
ax.set_title('D. Relative Growth (Sensitive)')

# Panel E: Endpoint dose-response (both populations)
ax = axes[1, 1]
for label, groups, color, marker in [
    ('Sensitive', sens_groups, '#2196F3', 'o'),
    ('Resistant', res_groups, '#F44336', 's'),
]:
    concs_list = sorted(groups.keys())
    mean_ep, std_ep = [], []
    for conc in concs_list:
        ep = [s.observations['cell_counts'][-1] for s in groups[conc]]
        mean_ep.append(np.mean(ep))
        std_ep.append(np.std(ep))
    # Normalize to control
    ctrl = mean_ep[0] if mean_ep[0] > 0 else 1
    ax.errorbar(np.array(concs_list) + 1e-3, np.array(mean_ep) / ctrl,
                yerr=np.array(std_ep) / ctrl, fmt=f'-{marker}', color=color,
                linewidth=2, markersize=5, capsize=3, label=label)
ax.set_xscale('symlog', linthresh=0.01)
ax.set_xlabel('Imatinib (uM)')
ax.set_ylabel('Relative viability (t=39h)')
ax.set_title('E. Endpoint Dose-Response')
ax.legend()

# Panel F: Empirical growth rates
ax = axes[1, 2]
for label, groups, color, marker in [
    ('Sensitive', sens_groups, '#2196F3', 'o'),
    ('Resistant', res_groups, '#F44336', 's'),
]:
    concs_list = sorted(groups.keys())
    for conc in concs_list:
        rates = []
        for s in groups[conc]:
            c = s.observations['cell_counts']
            t = s.times
            if c[-1] > 0 and c[0] > 0 and (t[-1] - t[0]) > 0:
                rates.append(np.log(c[-1] / c[0]) / (t[-1] - t[0]))
        if rates:
            ax.errorbar(conc + 1e-3, np.mean(rates), yerr=np.std(rates),
                       fmt=marker, color=color, markersize=5, capsize=3)
    ax.plot([], [], f'{marker}-', color=color, label=label)  # legend entry
ax.axhline(y=0, color='gray', linestyle=':', linewidth=0.8)
ax.set_xscale('symlog', linthresh=0.01)
ax.set_xlabel('Imatinib (uM)')
ax.set_ylabel('Apparent growth rate (1/h)')
ax.set_title('F. Empirical Net Growth Rate')
ax.legend()

fig.suptitle('PhenoPop Ba/F3 Data Overview', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig('../results/phenopop_validation/fig1_data_overview.png', dpi=200, bbox_inches='tight')
fig.savefig('../results/phenopop_validation/fig1_data_overview.pdf', bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

---
## 2. Fit Models

In [ ]:
# Per-concentration fitting: Sensitive
print('Fitting sensitive per-concentration...')
sens_per_conc = fit_per_concentration(sens_groups)

print('\nFitting resistant per-concentration...')
res_per_conc = fit_per_concentration(res_groups)

In [ ]:
# Full dose-response model: Sensitive
print('Fitting sensitive dose-response model...')
sens_dr = fit_dose_response(sens_groups)

print('\nSENSITIVE PARAMETERS:')
for name, val in sens_dr.parameters.items():
    se = sens_dr.se.get(name, float('nan')) if sens_dr.se else float('nan')
    print(f'  {name:<20s} = {val:10.5f}  (SE: {se:.5f})')

In [ ]:
# Full dose-response model: Resistant
print('Fitting resistant dose-response model...')
res_dr = fit_dose_response(res_groups)

print('\nRESISTANT PARAMETERS:')
for name, val in res_dr.parameters.items():
    print(f'  {name:<20s} = {val:10.5f}')

# Summary comparison
sp, rp = sens_dr.parameters, res_dr.parameters
print(f'\nEC50 ratio (Res/Sens): {rp.get("ec50_death", 0) / max(sp.get("ec50_death", 0), 1e-6):.1f}x')

---
## Figure 2: Model Fits and Mechanistic Dose-Response

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 11))
sp = sens_dr.parameters
rp = res_dr.parameters
c_range = np.logspace(-2.5, 1, 300)

# Build rate sets
sens_rs = build_rate_set(sp)
res_rs = build_rate_set(rp)
topo = ModelTopology.two_state()

# Panel A: Mechanistic dose-response curves (sensitive)
ax = axes[0, 0]
b_vals = [sens_rs.birth_rate(c) for c in c_range]
d_vals = [sens_rs.death_rate(CellType.P, c) for c in c_range]
net_vals = [b - d for b, d in zip(b_vals, d_vals)]

ax.semilogx(c_range, b_vals, '-', color='#2196F3', linewidth=2.5,
            label=f'Birth (b0={sp["b0"]:.4f}/h)')
ax.semilogx(c_range, d_vals, '-', color='#F44336', linewidth=2.5,
            label=f'Death (Emax={sp.get("emax_death",0):.4f})')
ax.semilogx(c_range, net_vals, '--', color='#333', linewidth=2, label='Net growth')
ax.axhline(y=0, color='gray', linestyle=':', linewidth=0.8)

# Mark EC50
ec50 = sp.get('ec50_death', 0)
ax.axvline(x=ec50, color='#F44336', linestyle=':', alpha=0.5)
ax.annotate(f'EC50={ec50:.2f}', xy=(ec50, max(d_vals)*0.8), fontsize=8, color='#F44336')

# Mark NG0
for c in c_range:
    if sens_rs.net_growth_rate(c) <= 0:
        ax.axvline(x=c, color='gray', linestyle='--', alpha=0.5)
        ax.annotate(f'NG0={c:.2f}', xy=(c, 0.002), fontsize=8, color='gray')
        break

ax.set_xlabel('Imatinib (uM)')
ax.set_ylabel('Rate (1/h)')
ax.set_title('A. Sensitive: Mechanistic Dose-Response')
ax.legend(fontsize=7)

# Panel B: Model vs data at select concentrations
ax = axes[0, 1]
select_concs = [0.0, 0.125, 0.375, 1.25, 5.0]
plot_colors = plt.cm.plasma(np.linspace(0, 0.9, len(select_concs)))

for conc_target, pc in zip(select_concs, plot_colors):
    ckey = min(s_concs, key=lambda c: abs(c - conc_target))
    if abs(ckey - conc_target) > 0.3:
        continue
    sl = sens_groups[ckey]
    all_c = [s.observations['cell_counts'] for s in sl]
    mean_c = np.mean(all_c, axis=0)
    std_c = np.std(all_c, axis=0)
    common_t = sl[0].times

    ax.errorbar(common_t, mean_c, yerr=std_c, fmt='o', color=pc,
               markersize=3, capsize=2, alpha=0.7)

    # Model prediction
    n0 = np.zeros(topo.n_states)
    n0[0] = mean_c[0]
    t_fine = np.linspace(0, common_t[-1], 100)
    try:
        ode = CellDynamicsODE(sens_rs, topo, lambda t, _c=ckey: _c)
        sim = ode.solve(n0, (0, t_fine[-1]), t_fine)
        ax.plot(t_fine, sim.viable, '-', color=pc, linewidth=2, label=f'{ckey} uM')
    except Exception as e:
        print(f'  Warning: C={ckey}: {e}')

ax.set_xlabel('Time (hours)')
ax.set_ylabel('Viable cell count')
ax.set_title('B. Model Predictions vs Data (Sensitive)')
ax.legend(title='[Imatinib]', fontsize=7, title_fontsize=8)

# Panel C: Sensitive vs resistant net growth comparison
ax = axes[0, 2]
s_net = [sens_per_conc[c].parameters.get('b0',0) - sens_per_conc[c].parameters.get('d0_P',0) for c in s_concs]
r_net = [res_per_conc[c].parameters.get('b0',0) - res_per_conc[c].parameters.get('d0_P',0) for c in r_concs]

# Also plot model curves
s_model_net = [sens_rs.net_growth_rate(c) for c in c_range]
r_model_net = [res_rs.net_growth_rate(c) for c in c_range]

ax.semilogx(c_range, s_model_net, 'b-', linewidth=2, alpha=0.5)
ax.semilogx(c_range, r_model_net, 'r-', linewidth=2, alpha=0.5)
ax.plot(np.array(s_concs) + 1e-3, s_net, 'bo', markersize=7, label='Sensitive (per-conc)', zorder=5)
ax.plot(np.array(r_concs) + 1e-3, r_net, 'rs', markersize=7, label='Resistant (per-conc)', zorder=5)
ax.axhline(y=0, color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('Imatinib (uM)')
ax.set_ylabel('Net growth rate (1/h)')
ax.set_title('C. Net Growth: Sensitive vs Resistant')
ax.legend(fontsize=8)

# Panel D: Residuals
ax = axes[1, 0]
all_res, all_conc_res = [], []
for conc in s_concs:
    sl = sens_groups[conc]
    all_c = [s.observations['cell_counts'] for s in sl]
    mean_c = np.mean(all_c, axis=0)
    common_t = sl[0].times
    n0 = np.zeros(topo.n_states)
    n0[0] = mean_c[0]
    try:
        ode = CellDynamicsODE(sens_rs, topo, lambda t, _c=conc: _c)
        sim = ode.solve(n0, (0, common_t[-1]), common_t)
        residuals = (mean_c - sim.viable) / np.maximum(mean_c, 1)
        all_res.extend(residuals)
        all_conc_res.extend([conc] * len(residuals))
    except Exception:
        pass

ax.scatter(np.array(all_conc_res) + 1e-3, all_res, alpha=0.5, s=20, color='#555')
ax.axhline(y=0, color='red', linestyle='-', linewidth=0.8)
ax.axhline(y=0.1, color='orange', linestyle=':', linewidth=0.5)
ax.axhline(y=-0.1, color='orange', linestyle=':', linewidth=0.5)
ax.set_xscale('symlog', linthresh=0.01)
ax.set_xlabel('Concentration (uM)')
ax.set_ylabel('Relative residual')
ax.set_title('D. Model Residuals (Sensitive)')

# Panel E: Parameter comparison table
ax = axes[1, 1]
ax.axis('off')
table_data = [
    ['Parameter', 'Sensitive', 'Resistant', 'Unit'],
    ['b0', f"{sp['b0']:.4f}", f"{rp['b0']:.4f}", '1/h'],
    ['d0_P', f"{sp['d0_P']:.5f}", f"{rp['d0_P']:.5f}", '1/h'],
    ['Emax_death', f"{sp.get('emax_death',0):.4f}", f"{rp.get('emax_death',0):.4f}", '1/h'],
    ['EC50_death', f"{sp.get('ec50_death',0):.3f}", f"{rp.get('ec50_death',0):.3f}", 'uM'],
    ['Hill_death', f"{sp.get('hill_death',0):.2f}", f"{rp.get('hill_death',0):.2f}", ''],
    ['AIC', f"{sens_dr.aic:.1f}", f"{res_dr.aic:.1f}", ''],
    ['Doubling time', f"{np.log(2)/sp['b0']:.1f}h", f"{np.log(2)/rp['b0']:.1f}h", ''],
]
table = ax.table(cellText=table_data, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.auto_set_column_width(col=[0, 1, 2, 3])
for j in range(4):
    table[0, j].set_facecolor('#4472C4')
    table[0, j].set_text_props(color='white', fontweight='bold')
ax.set_title('E. Parameter Comparison', fontsize=11)

# Panel F: AIC model comparison bar
ax = axes[1, 2]
# Also fit cytostatic for sensitive
print('Fitting cytostatic model for comparison...')
cs_topo = ModelTopology.two_state()
cs_names = ['b0', 'd0_P', 'emax_birth', 'ec50_birth', 'hill_birth',
            'u_PQ', 'u_QP', 'overdispersion']
cs_series = []
for sl in sens_groups.values():
    cs_series.extend(sl[:3])
cs_obs = CellCountObservation(overdispersion=10.0)
cs_lik = ModelLikelihood(topology=cs_topo, data=cs_series, param_names=cs_names,
                         mode='ode', observation_model=cs_obs)
cs_est = MLEstimator(cs_lik, bounds={
    'b0': (0.01, 0.12), 'd0_P': (1e-4, 0.05), 'emax_birth': (0.01, 0.99),
    'ec50_birth': (0.01, 5.0), 'hill_birth': (0.3, 5.0), 'u_PQ': (1e-5, 0.02),
    'u_QP': (1e-5, 0.02), 'overdispersion': (2.0, 500.0)}, method='L-BFGS-B')
cs_result = cs_est.fit(
    initial_guess=np.array([0.04, 0.005, 0.5, 0.5, 1.5, 0.003, 0.002, 50.0]),
    n_restarts=5)

labels = ['Cytotoxic\n(death increase)', 'Cytostatic\n(birth decrease)']
aics = [sens_dr.aic, cs_result.aic]
bar_colors = ['#F44336', '#2196F3']
bars = ax.bar(labels, aics, color=bar_colors, alpha=0.8, edgecolor='black')
best = np.argmin(aics)
bars[best].set_edgecolor('gold')
bars[best].set_linewidth(3)
ax.set_ylabel('AIC (lower is better)')
ax.set_title(f'F. Model Comparison (ΔAIC={abs(aics[1]-aics[0]):.1f})')

fig.suptitle('U-MIMIC Model Fits: PhenoPop Ba/F3 + Imatinib', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig('../results/phenopop_validation/fig2_model_fits.png', dpi=200, bbox_inches='tight')
fig.savefig('../results/phenopop_validation/fig2_model_fits.pdf', bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

---
## Figure 3: Synthetic Parameter Recovery

In [ ]:
# Generate synthetic data with known parameters
np.random.seed(42)

true_params = {
    'b0': 0.05, 'd0_P': 0.008, 'emax_death': 0.06,
    'ec50_death': 0.5, 'hill_death': 1.5, 'u_PQ': 0.005,
    'u_QP': 0.003, 'overdispersion': 60.0,
}

syn_topo = ModelTopology.two_state()
syn_rs = RateSet(
    birth_base=true_params['b0'],
    death_base={CellType.P: true_params['d0_P'], CellType.Q: 0.004},
    death_modulation={CellType.P: EmaxHill(
        emax=true_params['emax_death'], ec50=true_params['ec50_death'],
        hill=true_params['hill_death'])},
    transition_base={
        (CellType.P, CellType.Q): true_params['u_PQ'],
        (CellType.Q, CellType.P): true_params['u_QP']},
)

syn_concs = [0.0, 0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0]
syn_times = np.array([0, 6, 12, 18, 24, 30, 36, 42, 48])
phi = true_params['overdispersion']
n_syn_reps = 3

syn_series = []
for conc in syn_concs:
    ode = CellDynamicsODE(syn_rs, syn_topo, lambda t, _c=conc: _c)
    sim = ode.solve(np.array([1000.0, 0.0]), (0, 48), syn_times)
    viable = sim.viable
    for i in range(n_syn_reps):
        noisy = np.array([
            np.random.negative_binomial(phi, phi / (phi + max(mu, 1)))
            for mu in viable], dtype=float)
        ts = TimeSeriesData.from_counts(syn_times, noisy, concentration=conc,
                                        group_id=f'syn_C{conc}_rep{i}')
        syn_series.append(ts)

print(f'Generated {len(syn_series)} synthetic time series')

# Fit
param_names_syn = list(true_params.keys())
syn_obs = CellCountObservation(overdispersion=10.0)
syn_lik = ModelLikelihood(topology=syn_topo, data=syn_series,
                          param_names=param_names_syn, mode='ode',
                          observation_model=syn_obs)
syn_est = MLEstimator(
    syn_lik,
    bounds={'b0': (0.01, 0.12), 'd0_P': (1e-4, 0.05),
            'emax_death': (0.001, 0.3), 'ec50_death': (0.01, 5.0),
            'hill_death': (0.3, 5.0), 'u_PQ': (1e-5, 0.02),
            'u_QP': (1e-5, 0.02), 'overdispersion': (2.0, 500.0)},
    method='L-BFGS-B')
syn_result = syn_est.fit(
    initial_guess=np.array([0.04, 0.005, 0.04, 0.8, 1.0, 0.003, 0.002, 40.0]),
    n_restarts=5)

print('\nRecovery results:')
print(f'{"Parameter":<18s} {"True":>10s} {"Estimated":>10s} {"Error%":>8s}')
print('-' * 48)
for name in ['b0', 'emax_death', 'ec50_death', 'hill_death']:
    t = true_params[name]
    e = syn_result.parameters.get(name, np.nan)
    err = abs(e - t) / t * 100
    print(f'{name:<18s} {t:10.4f} {e:10.4f} {err:7.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

syn_est_params = syn_result.parameters

# Panel A: True vs estimated dose-response curves
ax = axes[0]
rs_true = build_rate_set(true_params)
rs_est = build_rate_set(syn_est_params)

true_net = [rs_true.net_growth_rate(c) for c in c_range]
est_net = [rs_est.net_growth_rate(c) for c in c_range]
true_d = [rs_true.death_rate(CellType.P, c) for c in c_range]
est_d = [rs_est.death_rate(CellType.P, c) for c in c_range]

ax.semilogx(c_range, true_net, 'g--', linewidth=2.5, label='True net growth')
ax.semilogx(c_range, est_net, 'r-', linewidth=2, label='Estimated net growth')
ax.semilogx(c_range, true_d, 'g:', linewidth=2, label='True death')
ax.semilogx(c_range, est_d, 'r:', linewidth=1.5, label='Estimated death')
ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
ax.set_xlabel('Concentration')
ax.set_ylabel('Rate (1/h)')
ax.set_title('A. Synthetic: Dose-Response Recovery')
ax.legend(fontsize=8)

# Panel B: Parameter ratio bar chart
ax = axes[1]
key_params = ['b0', 'emax_death', 'ec50_death', 'hill_death']
ratios = [syn_est_params[k] / true_params[k] for k in key_params]
bar_c = ['#2196F3' if 0.7 < r < 1.3 else '#FF9800' for r in ratios]

ax.bar(range(len(key_params)), ratios, color=bar_c, alpha=0.8, edgecolor='black')
ax.axhline(y=1, color='green', linestyle='--', linewidth=2, label='True value')
ax.axhspan(0.7, 1.3, alpha=0.1, color='green', label='30% band')
ax.set_xticks(range(len(key_params)))
ax.set_xticklabels(['b0', 'Emax', 'EC50', 'Hill'], fontsize=10)
ax.set_ylabel('Estimated / True')
ax.set_title('B. Recovery Accuracy')
ax.set_ylim(0, 2.0)
ax.legend(fontsize=8)

# Panel C: True vs Estimated scatter
ax = axes[2]
all_true = [true_params[k] for k in key_params]
all_est = [syn_est_params[k] for k in key_params]
labels_scatter = ['b0', 'Emax', 'EC50', 'Hill']
scatter_colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for t, e, lab, sc in zip(all_true, all_est, labels_scatter, scatter_colors):
    ax.scatter(t, e, s=100, zorder=5, label=lab, color=sc, edgecolors='black')

lims = [min(min(all_true), min(all_est)) * 0.5,
        max(max(all_true), max(all_est)) * 1.5]
ax.plot(lims, lims, 'k--', linewidth=1, alpha=0.5, label='y = x')
ax.set_xlabel('True value')
ax.set_ylabel('Estimated value')
ax.set_title('C. True vs Estimated')
ax.legend(fontsize=8)

fig.suptitle('Synthetic Parameter Recovery Benchmark', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig('../results/phenopop_validation/fig3_parameter_recovery.png', dpi=200, bbox_inches='tight')
fig.savefig('../results/phenopop_validation/fig3_parameter_recovery.pdf', bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

---
## Summary Statistics

In [ ]:
print('=' * 60)
print('VALIDATION SUMMARY')
print('=' * 60)

print(f'\n1. SENSITIVE Ba/F3 (cytotoxic model):')
print(f'   Birth rate: {sp["b0"]:.4f}/h (doubling: {np.log(2)/sp["b0"]:.1f}h)')
print(f'   EC50: {sp.get("ec50_death",0):.3f} uM')
print(f'   Hill: {sp.get("hill_death",0):.2f}')
print(f'   AIC: {sens_dr.aic:.1f}')

print(f'\n2. RESISTANT Ba/F3:')
print(f'   EC50: {rp.get("ec50_death",0):.3f} uM')
print(f'   Resistance ratio: {rp.get("ec50_death",0)/max(sp.get("ec50_death",0),1e-6):.1f}x')

print(f'\n3. Model comparison (sensitive):')
print(f'   Cytotoxic AIC: {sens_dr.aic:.1f}')
print(f'   Cytostatic AIC: {cs_result.aic:.1f}')
print(f'   ΔAIC: {abs(cs_result.aic - sens_dr.aic):.1f}')

print(f'\n4. Synthetic recovery:')
for name in ['b0', 'emax_death', 'ec50_death', 'hill_death']:
    err = abs(syn_est_params[name] - true_params[name]) / true_params[name] * 100
    status = 'PASS' if err < 30 else 'WARN'
    print(f'   {name}: {err:.1f}% error [{status}]')

print(f'\n5. Biological plausibility:')
print(f'   Ba/F3 doubling: {np.log(2)/sp["b0"]:.1f}h (lit: 15-20h)')
print(f'   Imatinib EC50: {sp.get("ec50_death",0):.3f} uM (lit: 0.1-1.0 uM)')
print(f'   Mechanism: cytotoxic (consistent with imatinib)')

In [ ]:
print('\nAll figures saved to: results/phenopop_validation/')
print('  fig1_data_overview.png/.pdf')
print('  fig2_model_fits.png/.pdf')
print('  fig3_parameter_recovery.png/.pdf')